In [ ]:
%pip install pandas matplotlib seaborn wordcloud nltk scikit-learn

In [ ]:
# Cell 1: Setup and Load Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk

# تحميل أدوات NLTK الأساسية (تعمل مرة واحدة فقط)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# قراءة البيانات 
# ضع اسم الملف المرفوع إذا كان بجانب النوت بوك، أو مساره الكامل
file_path = "data/Laptop_Train_v2.csv"
df = pd.read_csv(r"data/Laptop_Train_v2.csv")

print("✅ تم تحميل البيانات بنجاح!")
print("شكل البيانات (عدد الأسطر والأعمدة):", df.shape)

# فلترة المشاعر للإبقاء على إيجابي، سلبي، ومحايد فقط (استبعاد conflict)
valid_polarities = ['positive', 'negative', 'neutral']
df = df[df['polarity'].isin(valid_polarities)].copy()
df = df.reset_index(drop=True)

# عرض أول 5 أسطر
display(df.head())

In [ ]:
# Cell 2: Exploratory Data Analysis (EDA) - Person 1
from wordcloud import WordCloud

print("1. Dataset Info:")
display(df.info())

print("\n2. Class Distribution (Polarity):")
display(df['polarity'].value_counts())

# رسم توزيع الفئات
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='polarity', hue='polarity', palette='viridis', legend=False)
plt.title('Distribution of Sentiment Polarities')
plt.show()

# تحليل طول النصوص بناءً على عمود Sentence
df['text_length'] = df['Sentence'].apply(lambda x: len(str(x).split()))
print(f"\n3. Text Length Analysis:\nMax length: {df['text_length'].max()} words")
print(f"Min length: {df['text_length'].min()} words")
print(f"Average length: {round(df['text_length'].mean(), 2)} words")

# عمل سحابة كلمات (Word Cloud) للنصوص الإيجابية
positive_text = " ".join(df[df['polarity'] == 'positive']['Sentence'].astype(str))
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(positive_text)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud for Positive Reviews')
plt.show()

In [ ]:
import nltk
nltk.download('punkt_tab')

In [ ]:
# Cell 3: Text Preprocessing Pipeline - Person 2 & 3
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english'))

# دالة التنظيف
def clean_and_tokenize(text):
    text = str(text).lower() # 1. Lowercasing
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # 2. إزالة الروابط
    text = re.sub(r'\@\w+|\#', '', text) # 3. إزالة المنشن والهاشتاج
    text = re.sub(r'[^a-zA-Z\s]', '', text) # 4. إزالة الأرقام وعلامات الترقيم
    
    # 5. Tokenization
    tokens = word_tokenize(text)
    
    # 6. Stopwords Removal
    cleaned_tokens = [word for word in tokens if word not in stop_words]
    
    return cleaned_tokens

# تطبيق الدالة وإنشاء عمود للكلمات النظيفة
df['cleaned_tokens'] = df['Sentence'].apply(clean_and_tokenize)

print("Text after Preprocessing (Tokens):")
display(df[['Sentence', 'cleaned_tokens']].head())

In [ ]:
# Cell 3: Text Preprocessing Pipeline - Person 2 & 3
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english'))

# دالة التنظيف
def clean_and_tokenize(text):
    text = str(text).lower() # 1. Lowercasing
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # 2. إزالة الروابط
    text = re.sub(r'\@\w+|\#', '', text) # 3. إزالة المنشن والهاشتاج
    text = re.sub(r'[^a-zA-Z\s]', '', text) # 4. إزالة الأرقام وعلامات الترقيم
    
    # 5. Tokenization
    tokens = word_tokenize(text)
    
    # 6. Stopwords Removal
    cleaned_tokens = [word for word in tokens if word not in stop_words]
    
    return cleaned_tokens

# تطبيق الدالة وإنشاء عمود للكلمات النظيفة
df['cleaned_tokens'] = df['Sentence'].apply(clean_and_tokenize)

print("Text after Preprocessing (Tokens):")
display(df[['Sentence', 'cleaned_tokens']].head())

In [ ]:
# Cell 4: Lexical Processing (Stemming vs Lemmatization) - Person 4
from nltk.stem import PorterStemmer, WordNetLemmatizer

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def apply_stemming(tokens):
    return [stemmer.stem(word) for word in tokens]

def apply_lemmatization(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

# المقارنة بين Stemming و Lemmatization
df['stemmed_tokens'] = df['cleaned_tokens'].apply(apply_stemming)
df['lemmatized_tokens'] = df['cleaned_tokens'].apply(apply_lemmatization)

# دمج الكلمات الناتجة من Lemmatization لجمل جاهزة
df['final_clean_text'] = df['lemmatized_tokens'].apply(lambda x: " ".join(x))

print("Comparison: Stemming vs Lemmatization")
display(df[['cleaned_tokens', 'stemmed_tokens', 'lemmatized_tokens']].head())

In [ ]:
# Cell 5: Feature Extraction & Saving - Person 5
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# 1. Bag of Words (BoW) للتجربة
bow_vectorizer = CountVectorizer(max_features=1000)
bow_matrix = bow_vectorizer.fit_transform(df['final_clean_text'])
print(f"Bag of Words Matrix Shape: {bow_matrix.shape}")

# 2. TF-IDF (المعتمد للمودل)
tfidf_vectorizer = TfidfVectorizer(max_features=1500)
tfidf_matrix = tfidf_vectorizer.fit_transform(df['final_clean_text'])
print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")

# استخراج الكلمات كأعمدة
feature_names = tfidf_vectorizer.get_feature_names_out()

# تحويل لمصفوفة
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)

# دمج الكلمات مع عمودي Aspect Term و polarity
final_dataset = pd.concat([tfidf_df, df[['Aspect Term', 'polarity']]], axis=1)

# حفظ الملف النهائي
output_filename = "NLP_Phase2_Ready_Dataset.csv"
final_dataset.to_csv(output_filename, index=False)

print(f"\n✅ تم تجهيز الداتا بنجاح وحفظها باسم: {output_filename}")
display(final_dataset.head())

In [ ]:
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

print("⏳ جاري قراءة بيانات الاختبار...")
# قراءة ملف الاختبار
file_path = r"data/Laptops_Test_Data_PhaseA.csv"
test_df = pd.read_csv(file_path)

# ⚠️ تم حذف جزء "فلترة المشاعر" لأن ملف الاختبار لا يحتوي على الإجابات أصلاً!

print("🧹 جاري تنظيف النصوص والمعالجة اللغوية...")
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def fully_clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    tokens = word_tokenize(text)
    cleaned_tokens = [word for word in tokens if word not in stop_words]
    
    # استخدام Lemmatization
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in cleaned_tokens]
    
    return " ".join(lemmatized_tokens)

# تطبيق الدالة على عمود النصوص
test_df['final_clean_text'] = test_df['Sentence'].apply(fully_clean_text)

print("💾 جاري حفظ بيانات الاختبار الجاهزة...")
# نحفظ الملف كاملاً مع إضافة عمود النصوص النظيفة
output_test_file = "Cleaned_Test_Data_PhaseA.csv"
test_df.to_csv(output_test_file, index=False)

print(f"✅ تمت العملية بنجاح! تم حفظ الملف باسم: {output_test_file}")
display(test_df.head())

In [ ]:
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

print("⏳ جاري قراءة بيانات الاختبار...")
# قراءة ملف الاختبار
file_path = r"data/Laptops_Test_Data_PhaseB.csv"
test_df = pd.read_csv(file_path)

# ⚠️ تم حذف جزء "فلترة المشاعر" لأن ملف الاختبار لا يحتوي على الإجابات أصلاً!

print("🧹 جاري تنظيف النصوص والمعالجة اللغوية...")
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def fully_clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    tokens = word_tokenize(text)
    cleaned_tokens = [word for word in tokens if word not in stop_words]
    
    # استخدام Lemmatization
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in cleaned_tokens]
    
    return " ".join(lemmatized_tokens)

# تطبيق الدالة على عمود النصوص
test_df['final_clean_text'] = test_df['Sentence'].apply(fully_clean_text)

print("💾 جاري حفظ بيانات الاختبار الجاهزة...")
# نحفظ الملف كاملاً مع إضافة عمود النصوص النظيفة
output_test_file = "Cleaned_Test_Data_PhaseB.csv"
test_df.to_csv(output_test_file, index=False)

print(f"✅ تمت العملية بنجاح! تم حفظ الملف باسم: {output_test_file}")
display(test_df.head())

In [ ]:
# ==========================================
# مشروع مساق NLP - Sentiment Analysis
# القائد: مالك والفريق
# ==========================================

# 📦 1. استدعاء جميع المكتبات المطلوبة
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from wordcloud import WordCloud
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# تحميل حزم NLTK الأساسية
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print("\n" + "="*50)
print("🚀 المرحلة الأولى: قراءة بيانات التدريب وتجهيزها")
print("="*50)

# 📥 2. قراءة بيانات التدريب
train_path = r"data/Laptop_Train_v2.csv"
train_df = pd.read_csv(train_path)

# فلترة المشاعر لاستبعاد الـ conflict
valid_polarities = ['positive', 'negative', 'neutral']
train_df = train_df[train_df['polarity'].isin(valid_polarities)].copy()
train_df = train_df.reset_index(drop=True)
print(f"✅ تم تحميل بيانات التدريب بنجاح. الحجم: {train_df.shape}")

# 🧹 3. دالة التنظيف والمعالجة اللغوية الموحدة
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def fully_clean_text(text):
    text = str(text).lower() # Lowercasing
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # إزالة الروابط
    text = re.sub(r'\@\w+|\#', '', text) # إزالة المنشن والهاشتاج
    text = re.sub(r'[^a-zA-Z\s]', '', text) # إزالة الأرقام والرموز
    
    tokens = word_tokenize(text) # Tokenization
    cleaned_tokens = [word for word in tokens if word not in stop_words] # إزالة كلمات التوقف
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in cleaned_tokens] # Lemmatization
    
    return " ".join(lemmatized_tokens)

print("⚙️ جاري تنظيف نصوص التدريب...")
train_df['clean_text'] = train_df['Sentence'].apply(fully_clean_text)

print("\n" + "="*50)
print("🧠 المرحلة الثانية: تدريب النماذج وتقييمها")
print("="*50)

# 🔀 4. تقسيم بيانات التدريب لغايات التقييم الداخلي (80% تدريب - 20% تقييم)
# ملاحظة للدكتورة: نقوم بهذا التقسيم لأن ملف الـ Test الأصلي لا يحتوي على الإجابات
X = train_df['clean_text']
y = train_df['polarity']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 🔢 5. تحويل النصوص لأرقام باستخدام TF-IDF
vectorizer = TfidfVectorizer(max_features=1500)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val) # لاحظ استخدمنا transform فقط هنا

# 🤖 6. تدريب مودل Support Vector Machine (SVM)
print("⚙️ جاري تدريب خوارزمية SVM...")
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_tfidf, y_train)

# 📊 7. تقييم المودل
y_pred = svm_model.predict(X_val_tfidf)
acc = accuracy_score(y_val, y_pred)
print(f"🎯 دقة المودل (SVM Accuracy): {acc * 100:.2f}%\n")

print("📋 تقرير الأداء التفصيلي (Classification Report):")
print(classification_report(y_val, y_pred))

# رسم مصفوفة الخطأ
cm = confusion_matrix(y_val, y_pred, labels=['positive', 'negative', 'neutral'])
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['positive', 'negative', 'neutral'], 
            yticklabels=['positive', 'negative', 'neutral'])
plt.title('Confusion Matrix - SVM Model')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

print("\n" + "="*50)
print("🔮 المرحلة الثالثة: التنبؤ بملف الاختبار الخارجي (Phase A)")
print("="*50)

# 📂 8. قراءة وتنظيف ملف الاختبار (الذي لا يحتوي على الإجابات)
test_path = r"data/Laptops_Test_Data_PhaseA.csv"
try:
    test_df = pd.read_csv(test_path)
    print("⚙️ جاري تنظيف نصوص الاختبار...")
    test_df['clean_text'] = test_df['Sentence'].apply(fully_clean_text)
    
    # 🔢 9. تحويل الكلمات لأرقام وتوقع المشاعر
    X_test_final = vectorizer.transform(test_df['clean_text'])
    test_predictions = svm_model.predict(X_test_final)
    
    # 💾 10. حفظ النتائج في ملف جديد
    test_df['Predicted_Polarity'] = test_predictions
    final_output_path = "PhaseA_Final_Predictions.csv"
    test_df.to_csv(final_output_path, index=False)
    
    print(f"✅ تم التنبؤ بجميع بيانات الاختبار بنجاح!")
    print(f"📁 تم حفظ النتائج النهائية في ملف: {final_output_path}")
    display(test_df[['Sentence', 'Predicted_Polarity']].head())
    
except FileNotFoundError:
    print("⚠️ لم يتم العثور على ملف الاختبار، يرجى التأكد من المسار.")